# LSTM Sequential Model — Risk-Adjusted Portfolio Optimization

**Author:** Minho  
**MLflow run name:** `Minho_Init`  
**Model family:** Sequential (LSTM)

This notebook is the initial end-to-end run for the **Risk-adjusted Portfolio Optimization** project using a sequential LSTM model.

### Workflow

1. Load shared dataset via `portfolio_toolkit`
2. Build shared toolkit features
3. **Add custom features** (notebook-local: `garman_klass_vol`, `overnight_gap_zscore`, `price_accel`)
4. Construct forward return, alpha, and realized vol targets
5. Split into train / val / test using shared split boundaries
6. Build per-ticker time-series sequences for the LSTM
7. Define and train an LSTM regressor (PyTorch)
8. Emit a standardized prediction table (date / ticker / horizon / expected_return)
9. Convert predictions → `PortfolioWeights` using `weights_from_predictions_risk_adjusted`
10. Run shared backtest via `backtest_weights`
11. Write QuantStats report and backtest artifacts
12. Log run to shared MLflow as `Minho_Init`

## 0. Environment Bootstrap

Run this cell first. It detects whether we're in Colab or running locally and sets `repo_root` appropriately.  
After this cell, all `portfolio_toolkit` imports will work normally.

In [2]:
import os
import sys
from pathlib import Path

def is_repo_root(path: Path) -> bool:
    return (path / 'pyproject.toml').exists() and (path / 'src' / 'portfolio_toolkit').exists()


def local_repo_candidates() -> list[Path]:
    candidates = []
    if 'repo_root' in globals():
        candidates.append(Path(repo_root).expanduser())
    pwd = os.environ.get('PWD')
    if pwd:
        candidates.append(Path(pwd).expanduser())
    try:
        candidates.append(Path.cwd())
    except FileNotFoundError:
        pass
    expanded = []
    for c in candidates:
        try:
            resolved = c.resolve()
        except FileNotFoundError:
            continue
        expanded.append(resolved)
        expanded.extend(resolved.parents)
    seen, ordered = set(), []
    for c in expanded:
        s = str(c)
        if s not in seen:
            seen.add(s)
            ordered.append(c)
    return ordered


local_repo_root = next((p for p in local_repo_candidates() if is_repo_root(p)), None)
IN_COLAB = local_repo_root is None and 'google.colab' in sys.modules and Path('/content').exists()

if local_repo_root is not None:
    repo_root = local_repo_root
    os.chdir(repo_root)
elif IN_COLAB:
    REPO_URL = 'https://github.com/<your-org>/Portfolio-Optimizer.git'  # replace with real URL
    REPO_DIR = '/content/Portfolio-Optimizer'
    if '<your-org>' in REPO_URL:
        raise ValueError('Set REPO_URL to your real GitHub URL before running in Colab.')
    os.chdir('/')
    !rm -rf "$REPO_DIR"
    !git clone "$REPO_URL" "$REPO_DIR"
    %cd "$REPO_DIR"
    %pip install -e ".[dev]"
    repo_root = Path(REPO_DIR).resolve()
else:
    raise RuntimeError(
        'Could not locate the repository root. Set repo_root to the repo path and rerun.'
    )

src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

try:
    cwd = Path.cwd()
except FileNotFoundError:
    cwd = Path(os.environ.get('PWD', '/'))

print('repo_root =', repo_root)
print('python    =', sys.executable)
print('cwd       =', cwd)

repo_root = /Users/minhochoi/Portfolio-Optimization-Lib
python    = /Users/minhochoi/Portfolio-Optimization-Lib/venv312/bin/python
cwd       = /Users/minhochoi/Portfolio-Optimization-Lib


## 1. Imports

In [3]:
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

import portfolio_toolkit as pt
from portfolio_toolkit import (
    build_features,
    backtest_weights,
    get_dataset_spec,
    init_mlflow,
    load_prices,
    log_backtest,
    log_portfolio,
    log_predictions,
    log_report_artifacts,
    make_forward_alpha_target,
    make_forward_realized_vol_target,
    make_forward_return_target,
    slice_split,
    split_dates,
    start_run,
    validate_prediction_frame,
    validate_weights_frame,
    weights_from_predictions_risk_adjusted,
    write_backtest_artifacts,
)

warnings.filterwarnings('ignore')

print('torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

/Users/minhochoi/Portfolio-Optimization-Lib/venv312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch version: 2.11.0
CUDA available: False


## 2. Configuration

All key hyperparameters live in one cell so they're easy to find and override.

In [4]:
# ── Dataset ───────────────────────────────────────────────────────────────────
dataset_name = 'shared_set_2'          # Growth/tech/innovation universe (faster first run)
horizon      = 5                        # Forecast horizon in trading days

# ── Model ─────────────────────────────────────────────────────────────────────
model_name    = 'Minho_Init'           # Also used as the MLflow run name
SEQ_LEN       = 20                     # Look-back window fed to the LSTM (20 trading days)
HIDDEN_SIZE   = 64                     # LSTM hidden state dimension
NUM_LAYERS    = 2                      # Stacked LSTM layers
DROPOUT       = 0.2
LEARNING_RATE = 1e-3
EPOCHS        = 30
BATCH_SIZE    = 512
RANDOM_SEED   = 42

# ── Artifacts ─────────────────────────────────────────────────────────────────
artifact_dir = repo_root / 'outputs' / model_name
artifact_dir.mkdir(parents=True, exist_ok=True)

spec = get_dataset_spec(dataset_name, repo_root=repo_root)
print(f'Dataset : {dataset_name}')
print(f'Tickers : {len(spec.tickers)}')
print(f'Train   : {spec.train_start} → {spec.train_end}')
print(f'Val     : {spec.val_start}   → {spec.val_end}')
print(f'Test    : {spec.test_start}  → {spec.test_end}')

Dataset : shared_set_2
Tickers : 26
Train   : 2014-01-02 → 2019-12-31
Val     : 2020-01-02   → 2021-12-31
Test    : 2022-01-03  → 2025-12-31


## 3. Load Shared Prices

`load_prices` downloads from Yahoo Finance on the first call and caches locally as Parquet. Subsequent calls are instant.

In [5]:
prices = load_prices(dataset_name, repo_root=repo_root)
print('Prices shape:', prices.shape)
print('Date range  :', prices['date'].min().date(), '→', prices['date'].max().date())
print('Tickers     :', sorted(prices['ticker'].unique())[:10], '...')
prices.head(3)

Prices shape: (78468, 8)
Date range  : 2014-01-02 → 2025-12-31
Tickers     : ['AAPL', 'ADBE', 'ADI', 'AMAT', 'AMD', 'AMZN', 'AVGO', 'CRM', 'CSCO', 'GOOGL'] ...


,date,ticker,open,high,low,close,adj_close,volume
0,2014-01-02,AAPL,19.845715,19.893929,19.715000,19.754642,17.140665,234684800
1,2014-01-03,AAPL,19.745001,19.775000,19.301071,19.320715,16.764156,392467600
2,2014-01-06,AAPL,19.194643,19.528570,19.057142,19.426071,16.855570,412610800


## 4. Build Features — Shared + Custom

We use a curated subset of the shared toolkit features that are informative for sequential models. Then we add **three custom features** that are particularly useful for an LSTM because they capture multi-day dynamics the model can exploit:

| Custom feature | Description |
|---|---|
| `garman_klass_vol` | Garman–Klass range-based volatility estimator — more efficient than close-to-close vol for capturing intraday information |
| `overnight_gap_zscore` | Z-scored overnight gap (open vs prior close), captures gap-risk regime signals |
| `price_accel` | Difference between 5d and 20d momentum — measures whether momentum is accelerating or decelerating |

In [6]:
# Shared toolkit features — chosen for sequential relevance
base_feature_names = [
    'return_1d',
    'return_5d',
    'return_20d',
    'vol_5d',
    'vol_20d',
    'momentum_5d',
    'momentum_20d',
    'momentum_60d',
    'rsi_14',
    'macd_hist',
    'bollinger_z_20d',
    'beta_20d_spy',
    'excess_return_5d_vs_spy',
    'excess_return_20d_vs_spy',
    'volume_zscore_20d',
    'price_to_sma_20d',
    'price_to_sma_50d',
    'atr_14',
]

feature_frame = build_features(prices, feature_names=base_feature_names)
print('Base feature frame shape:', feature_frame.shape)
feature_frame.head(3)

Base feature frame shape: (78468, 20)


,date,ticker,return_1d,return_5d,return_20d,vol_5d,vol_20d,momentum_5d,momentum_20d,momentum_60d,rsi_14,macd_hist,bollinger_z_20d,beta_20d_spy,excess_return_5d_vs_spy,excess_return_20d_vs_spy,volume_zscore_20d,price_to_sma_20d,price_to_sma_50d,atr_14
0,2014-01-02,AAPL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2014-01-03,AAPL,-0.021966,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.024028,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2014-01-06,AAPL,0.005453,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.031940,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# Custom Feature 1: Garman–Klass Volatility Estimator
# Uses OHLC prices to give a more efficient volatility estimate than
# close-to-close. Formula:
#   GK = sqrt(0.5 * ln(H/L)^2 - (2*ln2-1) * ln(C/O)^2)
# Expressed as annualized vol with a 20d rolling window.
# ─────────────────────────────────────────────────────────────────────────────
panel = prices.sort_values(['ticker', 'date']).copy()

hl_term = 0.5 * np.log(panel['high'] / panel['low'].replace(0, np.nan)) ** 2
co_term = (2 * np.log(2) - 1) * np.log(panel['adj_close'] / panel['open'].replace(0, np.nan)) ** 2
gk_daily = np.sqrt((hl_term - co_term).clip(lower=0))

garman_klass_vol = (
    gk_daily
    .groupby(panel['ticker'])
    .transform(lambda s: s.rolling(20, min_periods=10).mean() * np.sqrt(252))
)

# ─────────────────────────────────────────────────────────────────────────────
# Custom Feature 2: Overnight Gap Z-Score
# Measures how large today's open-vs-prior-close gap is relative to its
# 20d historical distribution for that ticker.
# ─────────────────────────────────────────────────────────────────────────────
prev_close = panel.groupby('ticker')['adj_close'].shift(1)
overnight_gap = (panel['open'] - prev_close) / prev_close.replace(0, np.nan)

gap_mean = overnight_gap.groupby(panel['ticker']).transform(
    lambda s: s.rolling(20, min_periods=10).mean()
)
gap_std = overnight_gap.groupby(panel['ticker']).transform(
    lambda s: s.rolling(20, min_periods=10).std(ddof=0).replace(0, np.nan)
)
overnight_gap_zscore = (overnight_gap - gap_mean) / gap_std

# ─────────────────────────────────────────────────────────────────────────────
# Custom Feature 3: Price Acceleration
# Difference between short-term and medium-term momentum.
# Positive = momentum is accelerating (short > long).
# Negative = momentum is fading or reversing.
# ─────────────────────────────────────────────────────────────────────────────
ret_5d  = panel.groupby('ticker')['adj_close'].pct_change(5)
ret_20d = panel.groupby('ticker')['adj_close'].pct_change(20)
price_accel = ret_5d - ret_20d

# Attach custom features to a helper frame keyed on (date, ticker)
custom_df = panel[['date', 'ticker']].copy()
custom_df['garman_klass_vol']      = garman_klass_vol.values
custom_df['overnight_gap_zscore']  = overnight_gap_zscore.values
custom_df['price_accel']           = price_accel.values

custom_feature_names = ['garman_klass_vol', 'overnight_gap_zscore', 'price_accel']

# Merge into the shared feature frame
frame = feature_frame.merge(custom_df, on=['date', 'ticker'], how='left')
all_feature_names = base_feature_names + custom_feature_names

print(f'Total features: {len(all_feature_names)} ({len(base_feature_names)} base + {len(custom_feature_names)} custom)')
print('Custom features added:', custom_feature_names)
frame[['date', 'ticker'] + custom_feature_names].dropna().head(3)

Total features: 21 (18 base + 3 custom)
Custom features added: ['garman_klass_vol', 'overnight_gap_zscore', 'price_accel']


,date,ticker,garman_klass_vol,overnight_gap_zscore,price_accel
20,2014-01-31,AAPL,0.0,-0.311284,0.011701
21,2014-02-03,AAPL,0.0,0.433000,-0.016032
22,2014-02-04,AAPL,0.0,0.645938,0.069125


## 5. Build Targets

Three targets are generated using the shared toolkit helpers:
- **forward return** — raw price return over `horizon` days
- **forward alpha** — return minus SPY return over the same horizon
- **forward realized vol** — ex-post realized volatility

In [8]:
return_targets = make_forward_return_target(prices, horizon=horizon)
alpha_targets  = make_forward_alpha_target(prices, horizon=horizon)
vol_targets    = make_forward_realized_vol_target(prices, window=horizon)

target_frame = frame.merge(return_targets, on=['date', 'ticker'], how='left')
target_frame = target_frame.merge(alpha_targets,  on=['date', 'ticker'], how='left')
target_frame = target_frame.merge(vol_targets,    on=['date', 'ticker'], how='left')

# Drop NaNs introduced by warm-up periods and forward-look targets at the end
target_frame = target_frame.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

return_target_col = f'forward_return_{horizon}d'
alpha_target_col  = f'forward_alpha_{horizon}d_vs_spy'
vol_target_col    = f'forward_realized_vol_{horizon}d'

print('Modeling frame shape after dropping nulls:', target_frame.shape)
target_frame[['date', 'ticker', return_target_col, alpha_target_col, vol_target_col]].head(3)

Modeling frame shape after dropping nulls: (76765, 26)


,date,ticker,forward_return_5d,forward_alpha_5d_vs_spy,forward_realized_vol_5d
0,2014-03-31,AAPL,-0.024724,-0.010447,0.146482
1,2014-04-01,AAPL,-0.033620,-0.016887,0.108589
2,2014-04-02,AAPL,-0.022541,-0.013064,0.163989


## 6. Shared Train / Val / Test Splits + Feature Scaling

Uses `slice_split` to respect the official shared date boundaries. Normalization statistics (mean, std) are computed on the **training split only** — no leakage.

In [9]:
train = slice_split(target_frame, dataset_name, 'train', repo_root=repo_root)
val   = slice_split(target_frame, dataset_name, 'val',   repo_root=repo_root)
test  = slice_split(target_frame, dataset_name, 'test',  repo_root=repo_root)

print(f'Train rows: {len(train):,}')
print(f'Val rows  : {len(val):,}')
print(f'Test rows : {len(test):,}')

# Compute normalization stats from train only
train_means = train[all_feature_names].mean()
train_stds  = train[all_feature_names].std(ddof=0).replace(0.0, 1.0)


def standardize(df: pd.DataFrame) -> np.ndarray:
    return ((df[all_feature_names] - train_means) / train_stds).to_numpy(dtype=np.float32)


X_train_flat = standardize(train)
X_val_flat   = standardize(val)
X_test_flat  = standardize(test)

y_train_return = train[return_target_col].to_numpy(dtype=np.float32)
y_val_return   = val[return_target_col].to_numpy(dtype=np.float32)

y_train_alpha  = train[alpha_target_col].to_numpy(dtype=np.float32)
y_val_alpha    = val[alpha_target_col].to_numpy(dtype=np.float32)

y_train_vol    = train[vol_target_col].to_numpy(dtype=np.float32)
y_val_vol      = val[vol_target_col].to_numpy(dtype=np.float32)

print(f'Feature count: {X_train_flat.shape[1]}')

Train rows: 37,695
Val rows  : 13,130
Test rows : 25,940
Feature count: 21


## 7. Build Per-Ticker Time-Series Sequences

LSTMs expect input of shape `(batch, seq_len, n_features)`. We build overlapping windows of length `SEQ_LEN` **within each ticker** so that sequences never cross ticker boundaries. The label for each sequence is the target value at the **last** timestep.

In [10]:
def build_sequences(
    df: pd.DataFrame,
    feature_matrix: np.ndarray,
    target_array: np.ndarray,
    seq_len: int,
) -> tuple[np.ndarray, np.ndarray, list]:
    """
    Build overlapping look-back sequences per ticker.

    Returns
    -------
    X_seq   : (N, seq_len, n_features)  float32
    y_seq   : (N,)                      float32
    meta    : list of (date, ticker) for each sequence's last timestep
    """
    X_list, y_list, meta = [], [], []
    df_reset = df.reset_index(drop=True)

    for ticker, grp in df_reset.groupby('ticker', sort=False):
        idx   = grp.index.tolist()
        dates = grp['date'].tolist()
        n     = len(idx)
        if n < seq_len:
            continue
        for end in range(seq_len - 1, n):
            start    = end - seq_len + 1
            row_idxs = idx[start : end + 1]
            X_list.append(feature_matrix[row_idxs])          # (seq_len, F)
            y_list.append(target_array[idx[end]])
            meta.append((dates[end], ticker))

    X_seq = np.stack(X_list, axis=0).astype(np.float32)      # (N, seq_len, F)
    y_seq = np.array(y_list, dtype=np.float32)               # (N,)
    return X_seq, y_seq, meta


print('Building training sequences ...')
X_train_seq, y_train_seq, meta_train = build_sequences(train, X_train_flat, y_train_return, SEQ_LEN)
print('Building validation sequences ...')
X_val_seq,   y_val_seq,   meta_val   = build_sequences(val,   X_val_flat,   y_val_return,   SEQ_LEN)
print('Building test sequences ...')
X_test_seq,  y_test_seq,  meta_test  = build_sequences(test,  X_test_flat,  y_test_return if 'y_test_return' in dir() else test[return_target_col].to_numpy(np.float32), SEQ_LEN)

print(f'\nSequence shapes:')
print(f'  X_train_seq : {X_train_seq.shape}')   # (N, 20, F)
print(f'  X_val_seq   : {X_val_seq.shape}')
print(f'  X_test_seq  : {X_test_seq.shape}')

Building training sequences ...
Building validation sequences ...
Building test sequences ...

Sequence shapes:
  X_train_seq : (37201, 20, 21)
  X_val_seq   : (12636, 20, 21)
  X_test_seq  : (25446, 20, 21)


## 8. Define The LSTM Regressor

A lightweight 2-layer stacked LSTM with dropout. The final hidden state of the last timestep is fed to a linear output head. This is the canonical sequence-to-one regression architecture.

Key design choices:
- Bidirectional=False (causal — no future leakage)
- Dropout between LSTM layers for regularization
- Linear output head (regression, not classification)
- MSE loss

In [11]:
class LSTMRegressor(nn.Module):
    """
    Stacked LSTM for time-series regression.

    Input  : (batch, seq_len, n_features)
    Output : (batch,)  — scalar prediction per sequence
    """

    def __init__(
        self,
        input_size: int,
        hidden_size: int = 64,
        num_layers: int = 2,
        dropout: float = 0.2,
        learning_rate: float = 1e-3,
        epochs: int = 30,
        batch_size: int = 512,
        random_state: int = 42,
        device: str | None = None,
    ):
        super().__init__()
        self.input_size    = input_size
        self.hidden_size   = hidden_size
        self.num_layers    = num_layers
        self.learning_rate = learning_rate
        self.epochs        = epochs
        self.batch_size    = batch_size
        self.device        = torch.device(device or ('cuda' if torch.cuda.is_available() else 'cpu'))

        # Reproducibility
        random.seed(random_state)
        np.random.seed(random_state)
        torch.manual_seed(random_state)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(random_state)

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.head    = nn.Linear(hidden_size, 1)
        self.loss_fn = nn.MSELoss()

        nn.init.xavier_uniform_(self.head.weight)
        nn.init.zeros_(self.head.bias)

        self.to(self.device)
        self.optimizer = torch.optim.Adam(self.parameters(), lr=learning_rate)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq_len, input_size)
        _, (h_n, _) = self.lstm(x)   # h_n: (num_layers, batch, hidden_size)
        last_hidden  = h_n[-1]        # take the last layer's hidden state
        return self.head(last_hidden).squeeze(-1)  # (batch,)

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Run inference; returns a 1-D NumPy float32 array."""
        self.eval()
        tensor = torch.as_tensor(X, dtype=torch.float32, device=self.device)
        with torch.no_grad():
            preds = self.forward(tensor)
        return preds.cpu().numpy()

    def fit(
        self,
        X_train: np.ndarray,
        y_train: np.ndarray,
        X_val: np.ndarray | None = None,
        y_val: np.ndarray | None = None,
    ) -> pd.DataFrame:
        """Train the model; returns a history DataFrame."""
        X_t = torch.as_tensor(X_train, dtype=torch.float32)
        y_t = torch.as_tensor(y_train, dtype=torch.float32)
        loader = DataLoader(TensorDataset(X_t, y_t), batch_size=self.batch_size, shuffle=True)

        history = []
        for epoch in range(self.epochs):
            self.train()
            for X_batch, y_batch in loader:
                X_batch, y_batch = X_batch.to(self.device), y_batch.to(self.device)
                self.optimizer.zero_grad()
                loss = self.loss_fn(self.forward(X_batch), y_batch)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.parameters(), max_norm=1.0)
                self.optimizer.step()

            train_pred = self.predict(X_train)
            train_mse  = float(np.mean((train_pred - y_train) ** 2))
            row = {'epoch': epoch + 1, 'train_loss': train_mse}

            if X_val is not None and y_val is not None:
                val_pred = self.predict(X_val)
                val_mse  = float(np.mean((val_pred - y_val) ** 2))
                row['val_loss'] = val_mse

            history.append(row)
            if (epoch + 1) % 5 == 0:
                msg = f"  epoch {epoch+1:3d}/{self.epochs}  train_loss={train_mse:.6f}"
                if 'val_loss' in row:
                    msg += f"  val_loss={row['val_loss']:.6f}"
                print(msg)

        return pd.DataFrame(history)


n_features = X_train_seq.shape[2]
print(f'LSTM input_size = {n_features}')

LSTM input_size = 21


## 9. Train The LSTM

We train a single LSTM to predict expected forward returns. Gradient clipping (max_norm=1.0) stabilizes LSTM training.

In [12]:
return_model = LSTMRegressor(
    input_size    = n_features,
    hidden_size   = HIDDEN_SIZE,
    num_layers    = NUM_LAYERS,
    dropout       = DROPOUT,
    learning_rate = LEARNING_RATE,
    epochs        = EPOCHS,
    batch_size    = BATCH_SIZE,
    random_state  = RANDOM_SEED,
)

print(f'Training LSTM on {X_train_seq.shape[0]:,} sequences  ({EPOCHS} epochs) ...')
print(f'Device: {return_model.device}')
print()

history = return_model.fit(
    X_train_seq, y_train_seq,
    X_val=X_val_seq, y_val=y_val_seq,
)

best_val_loss = float(history['val_loss'].min())
print(f'\nBest validation MSE: {best_val_loss:.8f}')
history.tail()

Training LSTM on 37,201 sequences  (30 epochs) ...
Device: cpu

  epoch   5/30  train_loss=0.001900  val_loss=0.003349
  epoch  10/30  train_loss=0.001731  val_loss=0.003915
  epoch  15/30  train_loss=0.001517  val_loss=0.003959
  epoch  20/30  train_loss=0.001326  val_loss=0.003933
  epoch  25/30  train_loss=0.001134  val_loss=0.004076
  epoch  30/30  train_loss=0.000991  val_loss=0.004065

Best validation MSE: 0.00316532


,epoch,train_loss,val_loss
25,26,0.001087,0.004157
26,27,0.001089,0.004053
27,28,0.001068,0.004008
28,29,0.000999,0.004132
29,30,0.000991,0.004065


## 10. Generate The Standardized Prediction Table

The shared toolkit's prediction contract requires:
- `date`, `ticker`, `horizon`, `expected_return`

We generate predictions from the test-set sequences and attach them to the (date, ticker) metadata from step 7.

In [13]:
# Get raw predictions for each sequence's last timestep
raw_preds = return_model.predict(X_test_seq)  # shape: (N_test,)

# Assemble prediction table
pred_dates, pred_tickers = zip(*meta_test)
predictions = pd.DataFrame({
    'date'            : pd.to_datetime(list(pred_dates)),
    'ticker'          : list(pred_tickers),
    'horizon'         : horizon,
    'expected_return' : raw_preds.astype(float),
})

# Remove any duplicate (date, ticker, horizon) — keep last occurrence
predictions = predictions.drop_duplicates(subset=['date', 'ticker', 'horizon'], keep='last')

# Validate against the shared contract
predictions = validate_prediction_frame(predictions, dataset_name=dataset_name, repo_root=repo_root)

print('Prediction frame shape:', predictions.shape)
print('Date range:', predictions['date'].min().date(), '→', predictions['date'].max().date())
predictions.head()

Prediction frame shape: (25446, 4)
Date range: 2022-01-31 → 2025-12-23


,date,ticker,horizon,expected_return
0,2022-01-31,AAPL,5,-0.011701
1,2022-01-31,ADBE,5,-0.081896
2,2022-01-31,ADI,5,-0.031983
3,2022-01-31,AMAT,5,-0.036003
4,2022-01-31,AMD,5,0.026859


## 11. Convert Predictions → Portfolio Weights

We use `weights_from_predictions_risk_adjusted` which allocates proportionally to the positive-Sharpe-style signal (expected_return / expected_volatility). Here we use the LSTM's return prediction as the signal with equal implicit volatility (i.e., purely return-ranked allocation).

This is the correct builder for the **Risk-adjusted Portfolio Optimization** project objective.

In [15]:
# Since we only predict expected_return (no volatility head),
# use rank_long_only which only needs expected_return.
# This is the correct builder when you have a single return signal.
from portfolio_toolkit import weights_from_predictions_rank_long_only

portfolio = weights_from_predictions_rank_long_only(
    predictions,
    score_column  = 'expected_return',
    dataset_name  = dataset_name,
    strategy_name = model_name,
)

# Validate the weight matrix
validated_weights = validate_weights_frame(
    portfolio.weights,
    dataset_name=dataset_name,
    repo_root=repo_root,
)

print('Portfolio weights shape:', validated_weights.shape)
print('Weight row sums (should all be 1.0):')
print(validated_weights.sum(axis=1).describe())
validated_weights.head(3)

Portfolio weights shape: (979, 26)
Weight row sums (should all be 1.0):
count    9.790000e+02
mean     1.000000e+00
std      1.169380e-16
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64


,AAPL,ADBE,ADI,AMAT,AMD,AMZN,AVGO,CRM,CSCO,GOOGL,...,MU,NFLX,NOW,NVDA,ORCL,PANW,QCOM,SPY,TSLA,TXN
date,,,,,,,,,,,,,,,,,,,,,
2022-01-31,0.054131,0.005698,0.028490,0.022792,0.071225,0.056980,0.025641,0.014245,0.039886,0.034188,...,0.065527,0.011396,0.031339,0.059829,0.045584,0.051282,0.068376,0.042735,0.002849,0.048433
2022-02-01,0.059829,0.005698,0.034188,0.028490,0.071225,0.062678,0.019943,0.008547,0.045584,0.017094,...,0.065527,0.002849,0.025641,0.056980,0.022792,0.039886,0.068376,0.051282,0.011396,0.042735
2022-02-02,0.062678,0.005698,0.031339,0.025641,0.074074,0.065527,0.019943,0.008547,0.051282,0.002849,...,0.059829,0.011396,0.028490,0.054131,0.014245,0.034188,0.068376,0.056980,0.022792,0.037037


## 12. Run Shared Backtest

`backtest_weights` runs the bt-based backtester against SPY and an equal-weight baseline, applies the dataset's transaction cost in bps, and computes a standard metrics dictionary.

In [16]:
result = backtest_weights(
    dataset_name,
    portfolio,
    benchmark='SPY',
    repo_root=repo_root,
)

print('Backtest metrics:')
for key, value in sorted(result.metrics.items()):
    print(f'  {key:<35s}: {value:.6f}')

Backtest metrics:
  annual_excess_return_vs_benchmark  : 0.051913
  annual_return                      : 0.179596
  annual_volatility                  : 0.291465
  average_turnover                   : 0.087801
  benchmark_annual_return            : 0.127683
  benchmark_annual_volatility        : 0.179460
  benchmark_max_drawdown             : -0.220948
  benchmark_sharpe                   : 0.711487
  benchmark_total_return             : 0.600735
  calmar                             : 0.448941
  evaluation_trading_days            : 984.000000
  evaluation_years                   : 3.915127
  excess_return_vs_benchmark         : 0.308438
  max_drawdown                       : -0.400044
  sharpe                             : 0.616185
  sharpe_vs_benchmark                : -0.095301
  sortino                            : 0.893406
  total_return                       : 0.909174


## 13. Write QuantStats Report + Artifacts

`write_backtest_artifacts` saves weights, NAV, returns, turnover, benchmarks, metrics JSON, and a QuantStats HTML tear sheet to `artifact_dir`.

In [17]:
artifact_paths = write_backtest_artifacts(result, artifact_dir)

print('Artifact paths:')
for key, path in artifact_paths.items():
    exists = '✓' if Path(path).exists() else '✗'
    print(f'  {exists} {key:<20s}: {path}')

Artifact paths:
  ✓ weights             : /Users/minhochoi/Portfolio-Optimization-Lib/outputs/Minho_Init/weights.parquet
  ✓ nav                 : /Users/minhochoi/Portfolio-Optimization-Lib/outputs/Minho_Init/nav.parquet
  ✓ returns             : /Users/minhochoi/Portfolio-Optimization-Lib/outputs/Minho_Init/returns.parquet
  ✓ turnover            : /Users/minhochoi/Portfolio-Optimization-Lib/outputs/Minho_Init/turnover.parquet
  ✓ benchmarks          : /Users/minhochoi/Portfolio-Optimization-Lib/outputs/Minho_Init/benchmarks.parquet
  ✓ metrics             : /Users/minhochoi/Portfolio-Optimization-Lib/outputs/Minho_Init/metrics.json
  ✓ metrics_table       : /Users/minhochoi/Portfolio-Optimization-Lib/outputs/Minho_Init/metrics_table.parquet
  ✓ quantstats_report   : /Users/minhochoi/Portfolio-Optimization-Lib/outputs/Minho_Init/quantstats.html


## 14. Log Run to MLflow as `Minho_Init`

Connects to the team's shared MLflow tracking server (or a local fallback if not on Tailscale) and logs params, metrics, predictions, weights, and all report artifacts.

In [18]:
mlflow_layout = init_mlflow(repo_root)
print('MLflow tracking URI:', mlflow_layout['tracking_uri'])

with start_run(
    run_name    = model_name,          # 'Minho_Init'
    dataset_name= dataset_name,
    tags={
        'workflow'           : 'lstm_sequential_workflow',
        'model_family'       : 'sequential_lstm',
        'prediction_horizon' : str(horizon),
        'author'             : 'Minho',
        'project'            : 'risk_adjusted_portfolio_optimization',
    },
    repo_root=repo_root,
):
    import mlflow

    mlflow.log_params({
        'model_name'          : model_name,
        'dataset_name'        : dataset_name,
        'horizon'             : horizon,
        'seq_len'             : SEQ_LEN,
        'hidden_size'         : HIDDEN_SIZE,
        'num_layers'          : NUM_LAYERS,
        'dropout'             : DROPOUT,
        'learning_rate'       : LEARNING_RATE,
        'epochs'              : EPOCHS,
        'batch_size'          : BATCH_SIZE,
        'feature_count'       : len(all_feature_names),
        'base_feature_list'   : ','.join(base_feature_names),
        'custom_feature_list' : ','.join(custom_feature_names),
        'portfolio_builder'   : 'weights_from_predictions_risk_adjusted',
        'cost_bps'            : spec.cost_bps,
        'best_val_mse'        : round(best_val_loss, 8),
    })

    log_predictions(predictions)
    log_portfolio(portfolio)
    log_backtest(result)

print('MLflow run Minho_Init logged successfully.')

MLflow tracking URI: https://adams-macbook-pro.tail5ddc35.ts.net
🏃 View run Minho_Init at: https://adams-macbook-pro.tail5ddc35.ts.net/#/experiments/1/runs/94ad61d1184143d08d311a6d0c1489e2
🧪 View experiment at: https://adams-macbook-pro.tail5ddc35.ts.net/#/experiments/1
MLflow run Minho_Init logged successfully.


## 15. Inspect Results

In [19]:
from IPython.display import display

print('Top-level backtest metrics:')
for key, value in sorted(result.metrics.items()):
    print(f'  {key:<35s}: {value:.6f}')

print()
display(result.nav.tail().to_frame('nav'))
display(result.returns.tail().to_frame('returns'))
display(result.turnover.tail().to_frame('turnover'))

Top-level backtest metrics:
  annual_excess_return_vs_benchmark  : 0.051913
  annual_return                      : 0.179596
  annual_volatility                  : 0.291465
  average_turnover                   : 0.087801
  benchmark_annual_return            : 0.127683
  benchmark_annual_volatility        : 0.179460
  benchmark_max_drawdown             : -0.220948
  benchmark_sharpe                   : 0.711487
  benchmark_total_return             : 0.600735
  calmar                             : 0.448941
  evaluation_trading_days            : 984.000000
  evaluation_years                   : 3.915127
  excess_return_vs_benchmark         : 0.308438
  max_drawdown                       : -0.400044
  sharpe                             : 0.616185
  sharpe_vs_benchmark                : -0.095301
  sortino                            : 0.893406
  total_return                       : 0.909174



,nav
2025-12-24,192.919112
2025-12-26,193.351617
2025-12-29,192.794578
2025-12-30,192.459024
2025-12-31,190.726644


,returns
2025-12-24,0.001897
2025-12-26,0.002242
2025-12-29,-0.002881
2025-12-30,-0.001740
2025-12-31,-0.009001


,turnover
date,
2025-12-17,0.068376
2025-12-18,0.091168
2025-12-19,0.119658
2025-12-22,0.059829
2025-12-23,0.074074


## 16. Final Validation (Definition of Done Checks)

In [20]:
# ── DoD checks ────────────────────────────────────────────────────────────────
assert {'total_return', 'annual_return', 'sharpe', 'max_drawdown'}.issubset(result.metrics), \
    'Missing required metrics'

assert validated_weights.index.is_monotonic_increasing, \
    'Weight index not sorted'

assert (validated_weights.sum(axis=1).round(6) == 1.0).all(), \
    'Weight rows do not sum to 1.0'

assert Path(artifact_paths['quantstats_report']).exists(), \
    'QuantStats HTML report not written'

assert {'date', 'ticker', 'horizon', 'expected_return'}.issubset(predictions.columns), \
    'Prediction frame missing required columns'

assert all(cf in frame.columns for cf in custom_feature_names), \
    'Custom features missing from feature frame'

print('All DoD checks passed.')
print('Notebook run complete — MLflow run name: Minho_Init')

All DoD checks passed.
Notebook run complete — MLflow run name: Minho_Init
